In [1]:
!pip install --upgrade pip

In [2]:
!pip install duckdb

In [3]:
!pip install pyarrow

In [4]:
# Imports

from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import warnings

import matplotlib.pyplot as plt


In [5]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [6]:
warnings.filterwarnings('ignore')

In [7]:
# Paths

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "yelp_datasets" / "yelp_dataset"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_PATH = DATA_DIR / "yelp_academic_dataset_review.json"
BUSINESS_PATH = DATA_DIR / "yelp_academic_dataset_business.json"
TIP_PATH = DATA_DIR / "yelp_academic_dataset_tip.json"

In [8]:
%%time

# Read recent review data with DuckDB
# We keep the most recent 1M reviews to make the assignment manageable.

reviews = duckdb.sql(f"""
    SELECT
        review_id,
        user_id,
        business_id,
        stars AS review_stars,
        text AS review_text,
        date AS review_date
    FROM read_json_auto('{REVIEW_PATH}')
    ORDER BY date DESC
    LIMIT 1000000 
""").df()

reviews["review_date"] = pd.to_datetime(reviews["review_date"])

reviews.shape

CPU times: user 5.34 s, sys: 3.61 s, total: 8.95 s
Wall time: 5.47 s


(1000000, 6)

In [9]:
# Read business data

business = duckdb.sql(f"""
    SELECT
        business_id,
        name AS business_name,
        city,
        state,
        categories
    FROM read_json_auto('{BUSINESS_PATH}')
""").df()

business.shape

(150346, 5)

In [10]:
# Read tips data
# Tips are additional free-text context. We will be careful not to use future tips from the test period.

tips = duckdb.sql(f"""
    SELECT
        business_id,
        text AS tip_text,
        date AS tip_date,
        compliment_count AS tip_compliment_count
    FROM read_json_auto('{TIP_PATH}')
""").df()

tips["tip_date"] = pd.to_datetime(tips["tip_date"])

tips.shape

(908915, 4)

In [11]:
# Create sentiment target from review stars

reviews["target_sentiment"] = np.where(
    reviews["review_stars"] <= 2,
    "negative",
    np.where(reviews["review_stars"] == 3, "neutral", "positive")
)

reviews["target_sentiment"].value_counts(normalize=True)

target_sentiment
positive   0.66
negative   0.27
neutral    0.07
Name: proportion, dtype: float64

In [12]:
# Chronological split
# This avoids training on newer reviews and testing on older reviews.

reviews = reviews.sort_values("review_date").reset_index(drop=True)

split_idx = int(len(reviews) * 0.8)

train_reviews = reviews.iloc[:split_idx].copy()
test_reviews = reviews.iloc[split_idx:].copy()

train_end_date = train_reviews["review_date"].max()

train_reviews.shape, test_reviews.shape, train_end_date

((800000, 7), (200000, 7), Timestamp('2021-09-14 16:29:22'))

In [13]:
# Aggregate tips using only information available during the training period
# This prevents leakage from future tips.

tips_train_context = tips[tips["tip_date"] <= train_end_date].copy()

tips_agg = (
    tips_train_context
    .groupby("business_id")
    .agg(
        tip_count=("tip_text", "count"),
        sample_tips=("tip_text", lambda x: " | ".join(x.dropna().astype(str).head(3))),
        avg_tip_compliment=("tip_compliment_count", "mean")
    )
    .reset_index()
)

tips_agg.shape

(105245, 4)

In [14]:
# Merge train and test separately

def build_model_frame(review_df, business_df, tips_agg_df):
    """
    Build the modeling dataframe from review, business, and tip context.
    """
    model_df = (
        review_df
        .merge(business_df, on="business_id", how="left")
        .merge(tips_agg_df, on="business_id", how="left")
    )

    return model_df


train_df = build_model_frame(train_reviews, business, tips_agg)
test_df = build_model_frame(test_reviews, business, tips_agg)

train_df.shape, test_df.shape

((800000, 14), (200000, 14))

In [15]:
%%time

# Basic feature engineering
# These features are derived only from input text, not from the target.

def add_basic_features(df):
    """
    Add deterministic features that do not use target information.
    """
    df = df.copy()

    df["review_text"] = df["review_text"].fillna("").astype(str)
    df["categories"] = df["categories"].fillna("").astype(str)
    df["sample_tips"] = df["sample_tips"].fillna("").astype(str)
    df["city"] = df["city"].fillna("unknown").astype(str)
    df["state"] = df["state"].fillna("unknown").astype(str)

    df["review_word_count"] = df["review_text"].str.count(r"\S+").astype("int32")
    df["review_char_count"] = df["review_text"].str.len().astype("int32")
    df["tip_count"] = df["tip_count"].fillna(0).astype("int32")
    df["avg_tip_compliment"] = df["avg_tip_compliment"].fillna(0).astype("float32")

    return df


train_df = add_basic_features(train_df)
test_df = add_basic_features(test_df)

CPU times: user 10.7 s, sys: 83.1 ms, total: 10.8 s
Wall time: 10.8 s


In [16]:
%%time

# Create different text variants for ablation experiments

def add_text_variants(df):
    """
    Create text variants to test whether categories, tips, and location help.
    """
    df = df.copy()

    df["text_review_only"] = df["review_text"]

    df["text_review_categories"] = (
        "Review: " + df["review_text"] +
        "\nCategories: " + df["categories"]
    )

    df["text_review_categories_tips"] = (
        "Review: " + df["review_text"] +
        "\nCategories: " + df["categories"] +
        "\nTips: " + df["sample_tips"]
    )

    df["text_full_context"] = (
        "Review: " + df["review_text"] +
        "\nCategories: " + df["categories"] +
        "\nTips: " + df["sample_tips"] +
        "\nLocation: " + df["city"] + ", " + df["state"]
    )

    return df


train_df = add_text_variants(train_df)
test_df = add_text_variants(test_df)

CPU times: user 3.22 s, sys: 742 ms, total: 3.96 s
Wall time: 3.96 s


In [17]:
# Keep only selected modeling columns
# We intentionally drop leakage-prone fields such as business_stars, user_average_stars, review_useful, and rating_gap.

selected_columns = [
    "review_id",
    "business_id",
    "user_id",
    "review_date",
    "review_stars",
    "target_sentiment",
    "review_text",
    "text_review_only",
    "text_review_categories",
    "text_review_categories_tips",
    "text_full_context",
    "business_name",
    "city",
    "state",
    "categories",
    "sample_tips",
    "review_word_count",
    "review_char_count",
    "tip_count",
    "avg_tip_compliment"
]

train_model_df = train_df[selected_columns].copy()
test_model_df = test_df[selected_columns].copy()

train_model_df.shape, test_model_df.shape

((800000, 20), (200000, 20))

In [ ]:
%%time
# Save processed datasets

bert_train_model_df = train_model_df.astype({
    col: "object"
    for col in train_model_df.select_dtypes(include=["string"]).columns
})

bert_test_model_df = test_model_df.astype({
    col: "object"
    for col in test_model_df.select_dtypes(include=["string"]).columns
})

bert_train_model_df.to_pickle(PROCESSED_DIR / "bert_train_model_df.pkl")
bert_test_model_df.to_pickle(PROCESSED_DIR / "bert_test_model_df.pkl")

In [18]:
%%time
# Save processed datasets

train_model_df.to_pickle(PROCESSED_DIR / "train_model_df.pkl")
test_model_df.to_pickle(PROCESSED_DIR / "test_model_df.pkl")

print("Saved train and test datasets.")

Saved train and test datasets.
CPU times: user 2.34 s, sys: 766 ms, total: 3.1 s
Wall time: 4.14 s
